In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\bhara\OneDrive\文档\AIRFLY_project\data\flights.csv",low_memory=False)
df.head()


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# checks size
df.shape


(5819079, 31)

In [3]:
# checks dtypes and null
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              object 
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          object 
 7   ORIGIN_AIRPORT       object 
 8   DESTINATION_AIRPORT  object 
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24

In [4]:
# check missing values
df.isnull().sum().sort_values(ascending=False).head(10)


CANCELLATION_REASON    5729195
LATE_AIRCRAFT_DELAY    4755640
WEATHER_DELAY          4755640
AIRLINE_DELAY          4755640
AIR_SYSTEM_DELAY       4755640
SECURITY_DELAY         4755640
ELAPSED_TIME            105071
AIR_TIME                105071
ARRIVAL_DELAY           105071
WHEELS_ON                92513
dtype: int64

In [5]:
# clean delay columns
delay_cols = [
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    "AIRLINE_DELAY",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY"
]
df[delay_cols] = df[delay_cols].fillna(0)
df[delay_cols].isnull().sum()



LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0
AIRLINE_DELAY          0
AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
dtype: int64

In [6]:
# cancellation
df.loc[
    (df["CANCELLED"] == 1) & (df["CANCELLATION_REASON"].isnull()),"CANCELLATION_REASON"] = "Unknown"
df.loc[df["CANCELLED"] == 1, "CANCELLATION_REASON"].isnull().sum()



np.int64(0)

In [7]:
# feature engineering

# create route
df["ROUTE"] = df["ORIGIN_AIRPORT"] + "-" + df["DESTINATION_AIRPORT"]

# month name
df["MONTH_NAME"] = pd.to_datetime(df["MONTH"], format='%m').dt.month_name()

# weekday name
df["DAY_NAME"] = df["DAY_OF_WEEK"].map({
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
    7: "Sunday"
})

# departure hour
df["DEP_HOUR"] = df["SCHEDULED_DEPARTURE"] // 100


In [8]:
df[["ROUTE", "MONTH_NAME", "DAY_NAME", "DEP_HOUR"]].head()

,ROUTE,MONTH_NAME,DAY_NAME,DEP_HOUR
0,ANC-SEA,January,Thursday,0
1,LAX-PBI,January,Thursday,0
2,SFO-CLT,January,Thursday,0
3,LAX-MIA,January,Thursday,0
4,SEA-ANC,January,Thursday,0


In [9]:
# time conversion
def convert_time(col):
    return pd.to_datetime(
        df[col].astype(str).str.zfill(4),
        format="%H%M",
        errors="coerce"
    ).dt.time

time_cols = [
    "SCHEDULED_DEPARTURE",
    "SCHEDULED_ARRIVAL"
]

for col in time_cols:
    df[col + "_TIME"] = convert_time(col)


In [10]:
# delayed
df["IS_DELAYED"] = df["ARRIVAL_DELAY"].apply(
    lambda x: 1 if x > 15 else 0
)


In [11]:
df["IS_DELAYED"].value_counts()
df[["ARRIVAL_DELAY", "IS_DELAYED"]].head(10)

,ARRIVAL_DELAY,IS_DELAYED
0,-22.0,0
1,-9.0,0
2,5.0,0
3,-9.0,0
4,-21.0,0
5,8.0,0
6,-17.0,0
7,-10.0,0
8,-13.0,0
9,-15.0,0


In [12]:
cat_cols = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "CANCELLATION_REASON"
]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
df[cat_cols].head()



,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,CANCELLATION_REASON
0,AS,ANC,SEA,NAN
1,AA,LAX,PBI,NAN
2,US,SFO,CLT,NAN
3,AA,LAX,MIA,NAN
4,AS,SEA,ANC,NAN


In [13]:
removecol= [
    # Technical / Not needed,
    "TAIL_NUMBER"

    # Empty wheels data (raw + converted)
    "WHEELS_OFF",
    "WHEELS_ON"

    "SCHEDULED_DEPARTURE",
    "SCHEDULED_ARRIVAL"
]
df = df.drop(columns=removecol, errors="ignore")
# df.columns

In [14]:
df.to_csv(r"C:\Users\bhara\OneDrive\文档\AIRFLY_project\data\flights cleaned.csv", index=False)

In [15]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\bhara\OneDrive\文档\AIRFLY_project\data\flights cleaned.csv",low_memory=False)
df.shape

(5819079, 37)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 37 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   YEAR                      int64  
 1   MONTH                     int64  
 2   DAY                       int64  
 3   DAY_OF_WEEK               int64  
 4   AIRLINE                   object 
 5   FLIGHT_NUMBER             int64  
 6   TAIL_NUMBER               object 
 7   ORIGIN_AIRPORT            object 
 8   DESTINATION_AIRPORT       object 
 9   SCHEDULED_DEPARTURE       int64  
 10  DEPARTURE_TIME            float64
 11  DEPARTURE_DELAY           float64
 12  TAXI_OUT                  float64
 13  WHEELS_OFF                float64
 14  SCHEDULED_TIME            float64
 15  ELAPSED_TIME              float64
 16  AIR_TIME                  float64
 17  DISTANCE                  int64  
 18  WHEELS_ON                 float64
 19  TAXI_IN                   float64
 20  ARRIVAL_TIME            